In [ ]:
type Thin = list[bool]
def dom(f : Thin): # domain is big side
    return len(f)
def cod(f : Thin): # codomain is small side
    return sum(f)
# The action on contexts and terms
# f(t)    widens t
# (ctx)f  thins a context
def comp(f : Thin, g : Thin) -> Thin:
    assert cod(f) == dom(g)
    i = 0 
    result = []
    for a in f:
        if a:
            result.append(g[i])
            i += 1
        else:
            result.append(False)
    assert i == len(g)
    return result   

def lcm(f : Thin, g : Thin) -> Thin:
    assert len(f) == len(g)
    return [a and b for a,b in zip(f,g)]

def div(f : Thin, g : Thin) -> Thin:
    assert dom(f) == dom(g)
    assert all(not a for a,b in zip(f,g) if not b) # f is thinner than g
    return [a for a,b in zip(f,g) if b]
# comp(div(lcm(f,g), g), g) == f

type Id = tuple[Thin, int]
@dataclass
class ThinUF:
    parents : list[Id] = field(default_factory=list)
    def makeset(self, scope : int) -> Id:
        i = len(self.parents)
        id = ([True]*scope, i)
        self.parents.append(id)
        return id
    def find(self, x : Id) -> Id:
        thin, xid = x
        while True:
            (thiny, yid) = self.parents[xid]
            if xid == yid:
                assert all(thiny)
                return (thin, xid)
            thin = comp(thiny, thin)
            xid = yid
    def union(self, x : Id, y : Id) -> Id | None:
        thinx, xid = self.find(x)
        thiny, yid = self.find(y)
        if xid != yid or thinx != thiny:
            thinz = lcm(thinx,thiny)
            (_, z) = self.makeset(cod(thinz))
            self.parents[xid] = (div(thinz,thinx), z)
            self.parents[yid] = (div(thinz,thiny), z)
            return (thinz, z)
        else:
            # hmm
            return None

type Thin = tuple[bool, ...]
type Id = tuple[Thin, int]

def widen(thin, x : Id) -> Id: # weaken
    return (comp(thin,x[0]), x[1])




Yea, the case of identity thinning will be very common.
Maybe it does make sense to maintain an arena if thinning annotations
and a separate public slots vector.
This makes it much more incremental from context free
Bake in their rules on construction.


If the cod of x and y match, then the pullback becomes an equalizer which is more specialized.

Does this construction work on an category annotation?

is_eq(x,y) -> z,px,py | None
looks up rather than asserts pullback exists
but do we need to be doing lca?



Confusion
Things is different contexts can "thin equal"

t1 -thin1> weakt1  = weakt2 <thin2- t2
These are the arguments to union.
But what do we really learn?

Do we learn that t1 is sensical in common context?


We define a tag z that lives in the common (bigger) context. But then finding can't make a canonical one of these. The arrows go the wrong way. There isn't a canonical weakening.

So Unodes should contain projections to the common

Do I need Id(widethin, ) since ids are actually thinned from their original.
Maybe if I lookup the native size of rawid, I can tell by whether the thinning has a cod or dom of it.
Hmm. I need widenings?

Maybe the z is a marker saying (there is at least one term in this eclass that can work in this context)
No. Because 
[x] |- x*0
[y] |- y*0

Hmm. No but these are rename identical. The 0 |- 0 case doesn't work because 0 is great

[x] |- x * 0 = v0 * 0
[x,y] |- y * 0 = v1 * 0
Yes this is fine (if it even makes sense) because we have a stronger one.


1 |- v0 * 0


0 |- 0 





In [ ]:
# I just throw away te lcm itself
# and div has a weird precondition.
def widest_common_thinning(x : Thin, y : Thin) -> tuple[int, Thin, Thin]:
    assert dom(x) == dom(y)
    size = sum([a and b for a,b in zip(x,y)])
    proj_x = [b for a,b in zip(x,y) if a] # has dom == cod(x)
    proj_y = [a for a,b in zip(x,y) if b] # has dom == cod(y)
    return size, proj_x, proj_y

In [ ]:
@dataclass(frozen=True)
class App:
    f : Node
    x : Node


type Node = App | Lam | Var 

@dataclass
class EGraph:
    table : dict[Node,int] = field(default_factory=dict)
    nodes : list[Node] = field(default_factory=list)
    uf : ThinningUF = field(default_factory=ThinningUF)



lcm = thin to intersection. Common subseqeucne. common thinning
least common thinning
widest common thinning. wct

(f / g) . g = f


Ok, need to put the plutonium hemispheres together.



Thinnings form a tree-like partial order
https://en.wikipedia.org/wiki/Tree_(set_theory)
a proof relevant one. That's what let's us be kind of online about it

It's a forset of disconnnected thinning trees

Ok so we could have a subterm and a _path_ as the proof relevant connection.
subterms modulo theories though?
An observation/command chain as a path and sub-coterm?

Do in rust?




partial orders tree like
thinnings are a kind of bastracted subterm relationship
subterm union find.
destructor chains 

generalization of binary search for tree like orders

serial parallel partial order

destructor chains
If then else chains.
bdd context.
c1


In [ ]:
:dep bitvec

In [ ]:
use std::iter::zip;
#[derive(Clone, Debug, Eq, Hash, PartialEq)]
struct Thin (Vec<bool>);
// Is there a lighter weight thing if it doesn't need to be resiable? Neh.
impl Thin {

fn id(n : usize) -> Self {
    let mut v = Vec::with_capacity(n);
    for _ in 0..n{
        v.push(true);
    }
    Thin(v)
}

fn dom(&self) -> usize {
    self.0.len()
}

fn cod(&self) -> usize {
    self.0.iter().filter(|i| **i).count()
}


fn comp(&self, small : &Thin) -> Thin{
    assert!(self.cod() == small.dom());
    let mut res = Vec::with_capacity(self.0.len());
    let mut j = 0;
    for b in self.0.iter() {
        if *b {
            res.push(small.0[j]);
            j+=1;
        } else {
         res.push(false);   
        }
    }
    Thin(res)
}

fn lcm(&self, other : &Thin) -> Thin {
    assert!(self.dom() == other.dom());
    Thin(zip(self.0.iter(), other.0.iter()).map(|(x,y)| *x && *y).collect())
}

}






In [ ]:
assert!(Thin::id(3).dom() == 3);
assert!(Thin::id(3).cod() == 3);
let t1 = Thin(vec![true, false, true]);
assert!(&Thin::id(3).comp(&t1) == &t1);
assert!(&t1.comp(&Thin::id(2)) == &t1);



In [ ]:
use std::collections::HashMap;
#[derive(Clone, Debug, Eq, Hash, PartialEq)]
struct Id(Thin, usize);


#[derive(Clone, Debug, Eq, Hash, PartialEq)]
struct Node {
    f : String,
    args : Vec<Id>,
    extra : i8
}


#[derive(Clone, Debug,Eq,PartialEq, Default)]
struct ThinEGraph {
    memo : HashMap<Node, Id>,
    nodes : Vec<Node>,
    parents : Vec<Id>
}

impl ThinEGraph {
    fn add_node(&mut self, ctx : usize, node : Node) -> Id {
        if let Some(id) = self.memo.get(&node) {
            id.clone()
        }
        else {
            let id = Id(Thin::id(ctx), self.parents.len());
            self.parents.push(id.clone());
            self.memo.insert(node.clone(), id.clone());
            self.nodes.push(node);
            id
        }
    }
    fn app(&mut self, ctx : usize, f : String, args : Vec<Id>) -> Id {
        assert!(args.iter().all(|a| a.0.dom() == ctx));
        self.add_node(ctx, Node {f, args, extra:0})
    }
    fn lam(&mut self, body : Id) -> Id {
        self.add_node(body.0.dom() + 1, Node {f : "lam".to_string(), args: vec![body], extra:-1})
    }
    fn var(&mut self, idx : usize) -> Id {
        self.add_node(1, Node {f: "var".to_string(), args: vec![], extra: 1})
    }
    
}



How in the world is rewriting going to make any sense?
Any rule that has a binder in it is screwed?
foo(lam x, F(x)) ->  


In [ ]:
#[derive(Clone, Debug, Eq, Hash, PartialEq)]
enum Term {
    App(String,Vec<Term>),
    Var(String),
    Lam(String, Box<Term>),
    EId(Id)
}
type UserId = (Vec<String>, Id)

impl ThinEGraph {
    fn add_term(&mut self, maxctx : &[String], term : &Term) -> UserId {
        match term {
            Var(v) => (vec![v], self.var())
            Lam(v, body) => {
                let (mut ctx, bodyid) = self.add_term(&vec![v].extend(maxctx), body);
                if ctx[0] == v {
                    ctx.remove(0);
                    (ctx, self.lam(bodyid))
                }else{
                    (ctx, self.lam(bodyid))
                }
            }
            App(f, args) => {
                // reconcile args
                let uids = args.iter().map(|a| self.add_term(maxctx, a)).collect();
                
                let mut new_args = Vec::with_capacity(args.len());
                for a in args.iter() {
                    let (ctx, (thin, rawid)) = self.add_term(ctx, a);

                }



            }

        }

    }
}



Error: cannot find type `EGraph` in this scope

In [ ]:
impl EGraph {
    fn subst(&mut self, t : Id, v : usize, s : Id) {
        // recursively replace t[s/v]
        // v gets bumped around as you recurse. This is kind of like 
        fn worker()
    }
}

Error: cannot find type `EGraph` in this scope

If I can do egg lamda examples, we'll call that success.
Is named that bad really?


In [ ]:
(0..3).collect::<Vec<usize>>()

[0, 1, 2]